# Bayesian Neural Ordinary Differential Equations in Julia

In this tutorial, we show how a Neural ODE built with Lux.jl and OrdinaryDiffEq.jl can be combined with Bayesian estimation libraries like AdvancedHMC.jl. This converts the Neural ODE to a Bayesian Neural ODE, which lets us estimate the error in the Neural ODE's estimation and forecasting. A working example of the Bayesian Neural ODE with the NUTS sampler is shown. For more details, have a look at this paper: https://arxiv.org/abs/2012.07244

## Libraries

In [ ]:
Pollis.packages("Lux", "OrdinaryDiffEq", "SciMLSensitivity", "ComponentArrays", "Mooncake", "AdvancedHMC", "MCMCChains", "StatsPlots")

In [ ]:
using Lux, OrdinaryDiffEq, SciMLSensitivity, ComponentArrays, Mooncake, AdvancedHMC, MCMCChains, StatsPlots, Random

rng = Xoshiro(1)

## Step 1: Get the data from the Spiral ODE example

In [ ]:
u0 = [2.0, 0.0]
datasize = 40
tspan = (0.0, 1.0)
tsteps = range(tspan[1], tspan[2]; length = datasize)

function trueODEfunc(du, u, p, t)
    true_A = [-0.1 2.0; -2.0 -0.1]
    du .= ((u .^ 3)' * true_A)'
end

prob_trueode = ODEProblem(trueODEfunc, u0, tspan)
ode_data = Array(solve(prob_trueode, Tsit5(); saveat = tsteps))

## Step 2: Define the Neural ODE architecture

Note that this step offers a lot of flexibility in the number of layers and the number of units in each layer. It is not necessarily true that a 100 unit architecture is better at prediction or forecasting than a 50 unit architecture. On the other hand, a complicated architecture can take a huge computational time without increasing performance.

In [ ]:
dudt2 = Chain(x -> x .^ 3, Dense(2 => 50, tanh), Dense(50 => 2))
ps, st = f64(Lux.setup(rng, dudt2))
p0 = ComponentArray(ps)
prob_neuralode = ODEProblem((u, p, t) -> first(dudt2(u, p, st)), u0, tspan)

## Step 3: Define the loss function for the Neural ODE
The sampler works on a plain vector of weights: `weights` gives them back the structure of the network.

In [ ]:
weights(theta) = ComponentArray(theta, getaxes(p0))
predict_neuralode(theta) = Array(solve(prob_neuralode, Tsit5(); p = weights(theta), saveat = tsteps))
loss_neuralode(theta) = sum(abs2, ode_data .- predict_neuralode(theta))

## Step 4: Integrate the Bayesian estimation workflow, as prescribed by the AdvancedHMC interface, with the Neural ODE defined above

The AdvancedHMC interface requires us to specify: (a) the Hamiltonian log density and its gradient, (b) the sampler and (c) the step size adaptor.

For the log density, we use the loss function. The theta * theta term denotes the use of Gaussian priors.

The user can make several modifications to Step 4: try different acceptance ratios, warmup samples and posterior samples. One can also use the Variational Inference (ADVI) framework, which doesn't work quite as well as NUTS. The SGLD (Stochastic Gradient Langevin Dynamics) sampler is seen to have a better performance than NUTS. Have a look at https://sebastiancallh.github.io/post/langevin/ for a quick introduction to SGLD.

In [ ]:
l(theta) = -loss_neuralode(theta) - sum(theta .* theta)

theta0 = collect(p0)
cache = Mooncake.prepare_gradient_cache(l, theta0)

# The log density and its gradient, by Mooncake
function dldtheta(theta)
    value, (_, gradient) = Mooncake.value_and_gradient!!(cache, l, theta)
    return value, copy(gradient)
end

metric = DiagEuclideanMetric(length(theta0))
h = Hamiltonian(metric, l, dldtheta)

We use the NUTS sampler with an acceptance ratio of 0.45 in this example. In addition, we use Nesterov Dual Averaging for the step size adaptation.

We sample using 100 warmup samples and 100 posterior samples. More give a better picture of the posterior, and take longer.

In [ ]:
integrator = Leapfrog(find_good_stepsize(h, theta0))
kernel = HMCKernel(Trajectory{MultinomialTS}(integrator, GeneralisedNoUTurn()))
adaptor = StanHMCAdaptor(MassMatrixAdaptor(metric), StepSizeAdaptor(0.45, integrator))

warmup = 100
draws = 100
samples, stats = sample(h, kernel, theta0, warmup + draws, adaptor, warmup; progress = false)
posterior = samples[(warmup + 1):end]
println(length(posterior), " posterior draws")

## Step 5: Plot diagnostics

A: Plot the chains and the autocorrelation of the first 5 parameters.

In [ ]:
samples_reduced = reduce(hcat, posterior)[1:5, :]
Chain_Spiral = Chains(reshape(permutedims(samples_reduced), (draws, 5, 1)))
plot(Chain_Spiral)

In [ ]:
autocorplot(Chain_Spiral)

B: Plot the retrodicted data: 300 paths from posterior draws, and the best fit.

In [ ]:
pl = scatter(tsteps, ode_data[1, :]; color = :red, label = "Data: Var1", xlabel = "t", title = "Spiral Neural ODE")
scatter!(pl, tsteps, ode_data[2, :]; color = :blue, label = "Data: Var2")

for k in 1:300
    resol = predict_neuralode(posterior[rand(rng, 1:draws)])
    plot!(pl, tsteps, resol[1, :]; alpha = 0.04, color = :red, label = "")
    plot!(pl, tsteps, resol[2, :]; alpha = 0.04, color = :blue, label = "")
end

losses = map(loss_neuralode, posterior)
idx = argmin(losses)
prediction = predict_neuralode(posterior[idx])

plot!(pl, tsteps, prediction[1, :]; color = :black, w = 2, label = "")
plot!(pl, tsteps, prediction[2, :]; color = :black, w = 2, label = "Best fit prediction", ylims = (-2.5, 3.5))

C: The contour plot, Var2 against Var1.

In [ ]:
pl = scatter(ode_data[1, :], ode_data[2, :]; color = :red, label = "Data", xlabel = "Var1", ylabel = "Var2", title = "Spiral Neural ODE")

for k in 1:300
    resol = predict_neuralode(posterior[rand(rng, 1:draws)])
    plot!(pl, resol[1, :], resol[2, :]; alpha = 0.04, color = :red, label = "")
end

plot!(pl, prediction[1, :], prediction[2, :]; color = :black, w = 2, label = "Best fit prediction", ylims = (-2.5, 3))